In [1]:
import pandas as pd
import glob
import numpy as np
from evcouplings.compare import DistanceMap
from copy import deepcopy
from scipy.stats import ks_2samp
import os
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import seaborn as sns

from evcouplings.visualize.pymol import (
    pymol_pair_lines, pymol_mapping
)
from sklearn.metrics import roc_curve
from sklearn import metrics

## Job submission code

In [2]:
job_string = """#!/bin/bash
#SBATCH -c 1                               # Request one core
#SBATCH -t 0-00:40:00                       # Runtime in D-HH:MM format
#SBATCH -p short                           # Partition to run in
#SBATCH --mem=1GB                       # Memory total in MiB (for all cores)
#SBATCH -o {4}/{3}/{0}/job_%j.out                 # File to which STDOUT will be written, including job ID (%j)
#SBATCH -e {4}/{3}/{0}/job_%j.err                 # File to which STDERR will be written, including job ID (%j)
                                           # You can change the filenames given with -o and -e to any filenames you'd like

# You can change hostname to any command you would like to run
python {4}/run_GeO.py {0} {1} {2} 10000 {3} {4}

"""
absolute_path = !pwd
absolute_path = absolute_path[0]

In [3]:
positive_controls = pd.read_csv("summary_data/02.structure_hits_filtered.csv", index_col=0)
NUM_SHUFFLES = 10000
output_path = "output"

print(absolute_path)
for idx, row in positive_controls.iterrows():
    uid = row.uid
    rv = row.rv
    entry = row.Entry
    
    print(uid, rv, entry)
    formatted = job_string.format(uid, rv, entry, output_path, absolute_path)
    #print(formatted)
    
    output_filename = f"{absolute_path}/{output_path}/{uid}/submit_job_10000_iteration.sh"
    with open(output_filename, "w") as of:
        of.write(formatted)
    os.system(f"sbatch {output_filename}")
    
    #break


/n/data1/hms/dbmi/farhat/anna/structure_annotation/2024_11_13_GeO
Y876_MYCTU Rv0876c P9WKR5
Submitted batch job 53042786
Y2076_MYCTU Rv2076c P9WLL3
Submitted batch job 53042787
Y037_MYCTU Rv0037c P9WJY1
Submitted batch job 53042789
Y1995_MYCTU Rv1995 P9WLP3
Submitted batch job 53042790
Y2012_MYCTU Rv2012 P9WLM3
Submitted batch job 53042792
Y2084_MYCTU Rv2084 P9WLK1
Submitted batch job 53042793
Y2265_MYCTU Rv2265 P9WLG3
Submitted batch job 53042795
Y036_MYCTU Rv0036c P9WM91
Submitted batch job 53042796
Y3766_MYCTU Rv3766 O69731
Submitted batch job 53042797
Y2091_MYCTU Rv2091c P9WLJ5
Submitted batch job 53042798
LUCA_MYCTU Rv3723 O69690
Submitted batch job 53042799
Y894_MYCTU Rv0894 P9WKP9
Submitted batch job 53042800
Y1269_MYCTU Rv1269c P9WM45
Submitted batch job 53042801
Y2570_MYCTU Rv2570 P9WL91
Submitted batch job 53042802
Y495_MYCTU Rv0495c P9WKU5
Submitted batch job 53042804
Y2600_MYCTU Rv2600 P9WFG5
Submitted batch job 53042806
Y049_MYCTU Rv0049 P9WM85
Submitted batch job 53042807

In [4]:
!squeue -u ag300 

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          53041510 interacti     bash    ag300  R      24:56      1 compute-a-17-61
          53043188     short submit_j    ag300 CG       0:49      1 compute-e-16-192
          53042951     short submit_j    ag300 CG       7:12      1 compute-e-16-241
          53043187     short submit_j    ag300 PD       0:00      1 (None)
          53044138     short submit_j    ag300 PD       0:00      1 (None)
          53044137     short submit_j    ag300 PD       0:00      1 (None)
          53044136     short submit_j    ag300 PD       0:00      1 (None)
          53044135     short submit_j    ag300 PD       0:00      1 (None)
          53044134     short submit_j    ag300 PD       0:00      1 (None)
          53044133     short submit_j    ag300 PD       0:00      1 (None)
          53044132     short submit_j    ag300 PD       0:00      1 (None)
          53044131     short submit_j    ag300 PD       0:00 

## Analysis of runs

In [5]:
# dataset = pd.read_csv("summary_data/02.structure_hits_filtered.csv", index_col=0)
# output_path= "output"

# problem_uids = []

# for idx, row in dataset.iterrows():
#     #print(row)
    
#     if not os.path.isfile(f"{output_path}/{row.uid}/G_scores.csv"):
#         print(row.uid)
#         problem_uids.append(row.uid)

        
# print(f"{len(problem_uids)} runs failed")

In [12]:
# positive_controls = pd.read_csv("summary_data/02.structure_hits_filtered.csv", index_col=0)
# NUM_SHUFFLES = 10000
# output_path = "output"

# positive_controls = positive_controls.query("uid in @problem_uids")

# print(absolute_path)
# for idx, row in positive_controls.iterrows():
#     uid = row.uid
#     rv = row.rv
#     entry = row.Entry
    
#     print(uid, rv, entry)
#     formatted = job_string.format(uid, rv, entry, output_path, absolute_path)
#     #print(formatted)
    
#     output_filename = f"{absolute_path}/{output_path}/{uid}/submit_job_10000_iteration.sh"
#     with open(output_filename, "w") as of:
#         of.write(formatted)
#     os.system(f"sbatch {output_filename}")
    
#     #break
